# RAG Pipeline — Method Statement Generator (RCC Works)

**Full pipeline:**
1. **Stage 1 — Parse** (`parser.py` v3): Extract text & tables from CPWD Specification PDF using PyMuPDF, build semantic chunks with rich metadata.
2. **Stage 2 — Embed & Store** (`embed_and_store.py` v4): Enrich chunks, generate embeddings (`all-MiniLM-L12-v2`), persist in ChromaDB.
3. **Stage 3 — Retrieve & Generate** (`retrieve_and_generate.py` v5): Multi-query retrieval with cross-encoder reranking + Groq LLM generation for each Method Statement section.
4. **Stage 4 — Generate DOCX**: Python-based DOCX generation from the final JSON (replaces `generate_docx.js`).

---
## Google Colab Setup Instructions

> **Before running:**
> 1. Upload your PDF (`Prescriptive Specifications_CPWD.pdf`) to Colab using the file upload widget in **Cell 3**.
> 2. Set your **Groq API Key** in the `GROQ_API_KEY` variable in the Configuration cell.
>    - Get a free key at: https://console.groq.com
> 3. Run cells **top-to-bottom** (`Runtime → Run all` or `Ctrl+F9`).
> 4. The final `.docx` file will be auto-downloaded at the end.
>
> **Runtime recommendation:** CPU is sufficient. GPU is not required.
> **Expected total runtime:** ~10–20 minutes (embedding + LLM calls).

---
## Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
# Note: bert-score is optional but recommended for evaluation metrics
!pip install -q pymupdf chromadb sentence-transformers groq python-docx bert-score python-dotenv

---
## Cell 2 — Imports

In [ ]:
import os
import re
import sys
import json
import shutil
from datetime import datetime
from typing import Optional

import fitz  # PyMuPDF
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from groq import Groq

# BERT Score — optional
try:
    from bert_score import score as bert_score_fn
    BERT_SCORE_AVAILABLE = True
except ImportError:
    BERT_SCORE_AVAILABLE = False
    print("⚠  bert-score not installed — BERT Score will be skipped.")

# python-docx
from docx import Document as DocxDocument
from docx.shared import Pt, RGBColor, Inches, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

print("✅ All imports successful")

---
## Cell 3 — Configuration & PDF Upload

> **Set your Groq API key below.** Then run the cell to upload your PDF.

In [ ]:
# ─────────────────────────────────────────────────────────────
# USER CONFIGURATION — edit these values
# ─────────────────────────────────────────────────────────────
GROQ_API_KEY    = "YOUR_GROQ_API_KEY_HERE"   # <-- paste your Groq key here
TEAM_NAME       = "Musketeers"
TEAM_MEMBERS    = [
    "Rishabh Sharma (Team Leader)",
    "Aman Likhitkar",
    "Rishabh Bharadwaj",
    "Deepak Pachauri",
]

# Paths
PDF_PATH        = "Prescriptive Specifications_CPWD.pdf"   # expected upload name
PARSED_JSON     = "parsed_sections.json"
DB_DIR          = "chroma_db"
OUT_DIR         = "output"
DOCX_OUT        = os.path.join(OUT_DIR, "Method_Statement_RCC.docx")

# Pipeline hyper-parameters (match original scripts)
EMBEDDING_MODEL_NAME = "all-MiniLM-L12-v2"
N_PER_QUERY          = 8       # chunks retrieved per query (v5 default)
DISTANCE_THRESHOLD   = 1.4
MAX_CONTEXT_CHARS    = 16000
RETRIEVAL_POOL_CAP   = 16

os.makedirs(OUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# PDF Upload widget (Colab)
# ─────────────────────────────────────────────────────────────
try:
    from google.colab import files
    print("Running in Google Colab — use the upload widget below to upload your PDF.")
    print(f"Expected filename: {PDF_PATH}")
    uploaded = files.upload()
    # Rename if uploaded with a different name
    for name in uploaded:
        if name != PDF_PATH:
            os.rename(name, PDF_PATH)
            print(f"Renamed '{name}' → '{PDF_PATH}'")
except ImportError:
    # Running locally — just check file exists
    if not os.path.exists(PDF_PATH):
        print(f"⚠  PDF not found at '{PDF_PATH}'. Place the file in the working directory.")
    else:
        print(f"✅ PDF found: {PDF_PATH}")

if os.path.exists(PDF_PATH):
    print(f"✅ PDF ready: {PDF_PATH}")

---
## Cell 4 — Stage 1: PDF Parser (parser.py v3)

Extracts text + tables from the PDF, builds a hierarchical section tree, applies semantic chunking with overlap, and outputs rich-metadata chunks.

In [ ]:
# ══════════════════════════════════════════════════════════════
# PARSER — Regex helpers
# ══════════════════════════════════════════════════════════════
SECTION_REGEX   = re.compile(r'^(\d+(?:\.\d+)+)\.?\s+(.*)')
TABLE_CAP_REGEX = re.compile(r'(TABLE|Table)\s+(\d+\.\d+)\b(.*)', re.IGNORECASE)


# ── Semantic Chunking Helpers ──────────────────────────────────
def get_hierarchy_path(node, parent_map: dict) -> str:
    path = [node.section_id]
    current = node
    while current.section_id in parent_map:
        current = parent_map[current.section_id]
        if current.section_id != "root":
            path.insert(0, current.section_id)
    return " > ".join(path)


def split_long_content(content: str, max_chunk_size: int = 800,
                       overlap_size: int = 150) -> list:
    if len(content) <= max_chunk_size:
        return [content]
    sentences = re.split(r'(?<=[.!?])\s+', content)
    chunks, current_chunk = [], ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) + 1 <= max_chunk_size:
            current_chunk += (" " + sentence if current_chunk else sentence)
        else:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = sentence
    if current_chunk:
        chunks.append(current_chunk)
    if len(chunks) > 1:
        overlapped = [chunks[0]]
        for i in range(1, len(chunks)):
            prev = chunks[i - 1]
            overlap_text = prev[-overlap_size:] if len(prev) > overlap_size else prev
            overlapped.append(overlap_text + " " + chunks[i])
        return overlapped
    return chunks


def generate_table_description(table_id: str, data: list) -> str:
    if not data:
        return f"{table_id}: (empty)"
    description = f"{table_id}:"
    if data:
        header = [h for h in data[0] if h.strip()]
        if header:
            description += f" Columns: {', '.join(header[:3])}"
            if len(header) > 3:
                description += f" (+{len(header) - 3} more)"
    non_empty_rows = sum(1 for row in data if any(c.strip() for c in row))
    description += f". Rows: ~{non_empty_rows}"
    return description


# ── Section Tree ───────────────────────────────────────────────
class SectionNode:
    def __init__(self, section_id: str, title: str):
        self.section_id = section_id
        self.title      = title
        self.content    = []
        self.tables     = []
        self.children   = []

    def add_content(self, text: str): self.content.append(text)
    def add_table(self, table: dict): self.tables.append(table)
    def to_dict(self):
        return {
            "section" : self.section_id,
            "title"   : self.title,
            "content" : " ".join(self.content).strip(),
            "tables"  : self.tables,
            "children": [c.to_dict() for c in self.children],
        }


def get_level(section_id: str) -> int:
    parts = section_id.split(".")
    return len(parts) - 2 if parts[-1] == "0" else len(parts) - 1


def find_parent(stack: list, section_id: str):
    current_level = get_level(section_id)
    for node in reversed(stack):
        if node.section_id == "root":
            return node
        if get_level(node.section_id) == current_level - 1:
            return node
    return stack[0]


def is_valid_section(text: str) -> bool:
    text = text.strip()
    if re.search(r'^\s*[\d\.\-\s()]+\s*(mm|cm|m|kg|%|mm²|m³|N\/mm²)\s*$', text.lower()):
        return False
    digit_ratio = sum(c.isdigit() for c in text) / max(len(text), 1)
    if digit_ratio > 0.4: return False
    if re.match(r'^[\d\.\-\s()]+$', text): return False
    if not re.search(r'[A-Za-z]', text): return False
    if len(text) < 5: return False
    if re.match(r'^(page|chapter|table of contents|index|appendix|glossary)', text.lower()):
        return False
    return True


# ── Geometry helpers ───────────────────────────────────────────
def rect_contains(outer, inner_point) -> bool:
    x0, y0, x1, y1 = outer
    px, py = inner_point
    return x0 - 2 < px < x1 + 2 and y0 - 2 < py < y1 + 2


def point_in_any_table(px, py, table_bboxes) -> bool:
    return any(rect_contains(bb, (px, py)) for bb in table_bboxes)


# ── Table caption helpers ──────────────────────────────────────
def clean_table_rows(raw_rows):
    cleaned = []
    for row in raw_rows:
        cleaned_row = []
        for cell in row:
            if cell is None:
                cleaned_row.append("")
            else:
                cleaned_row.append(str(cell).replace("\n", " ").strip())
        cleaned.append(cleaned_row)
    return cleaned


def extract_embedded_caption(rows):
    if not rows: return None
    first_non_empty = [c for c in rows[0] if c]
    if not first_non_empty: return None
    m = TABLE_CAP_REGEX.match(first_non_empty[0])
    if m:
        label  = f"Table {m.group(2)}"
        suffix = m.group(3).strip(" :-")
        return f"{label} {suffix}".strip() if suffix else label
    return None


def drop_embedded_caption_row(rows):
    if not rows: return rows
    first_non_empty = [c for c in rows[0] if c]
    if first_non_empty and TABLE_CAP_REGEX.match(first_non_empty[0]):
        return rows[1:]
    return rows


print("✅ Parser helpers defined")

In [ ]:
# ══════════════════════════════════════════════════════════════
# PARSER — Core extraction functions
# ══════════════════════════════════════════════════════════════
def extract_page_items(doc):
    items = []
    for page_num, page in enumerate(doc):
        tab_finder   = page.find_tables()
        page_tables  = tab_finder.tables
        table_bboxes = [t.bbox for t in page_tables]
        pending_caption = None

        text_data  = page.get_text("dict")
        text_items = []
        for block in text_data["blocks"]:
            if "lines" not in block: continue
            for line in block["lines"]:
                line_text = ""
                y_center  = (line["bbox"][1] + line["bbox"][3]) / 2
                x_left    = line["bbox"][0]
                for span in line["spans"]:
                    line_text += span["text"]
                line_text = line_text.strip()
                if not line_text: continue
                if point_in_any_table(x_left, y_center, table_bboxes): continue
                text_items.append((line["bbox"][1], line_text))

        ordered = []
        for y, txt in text_items:
            ordered.append((y, {"type": "text", "text": txt, "y": y, "page": page_num}))
        for tbl in page_tables:
            y_top = tbl.bbox[1]
            rows  = clean_table_rows(tbl.extract())
            cap   = extract_embedded_caption(rows)
            if cap:
                rows = drop_embedded_caption_row(rows)
            ordered.append((y_top, {"type": "table_pending", "embedded_cap": cap,
                                     "data": rows, "y": y_top, "page": page_num}))
        ordered.sort(key=lambda x: x[0])

        for _, item in ordered:
            if item["type"] == "text":
                m = TABLE_CAP_REGEX.match(item["text"])
                if m:
                    label  = f"Table {m.group(2)}"
                    suffix = m.group(3).strip(" :-")
                    pending_caption = f"{label} {suffix}".strip() if suffix else label
                    continue
                items.append(item)
            elif item["type"] == "table_pending":
                table_id = pending_caption or item["embedded_cap"] or "Table (unknown)"
                pending_caption = None
                items.append({"type": "table", "table_id": table_id,
                               "data": item["data"], "y": item["y"], "page": item["page"]})
    return items


def merge_split_tables(items):
    merged, i = [], 0
    while i < len(items):
        item = items[i]
        if item["type"] == "table" and i + 1 < len(items):
            j = i + 1
            while j < len(items) and items[j]["type"] == "text":
                j += 1
            if j < len(items):
                nxt = items[j]
                is_continuation = (
                    nxt["type"] == "table"
                    and "unknown" in nxt["table_id"].lower()
                    and nxt["page"] == item["page"] + 1
                    and len(nxt["data"]) > 0 and len(item["data"]) > 0
                    and len(nxt["data"][0]) == len(item["data"][0])
                )
                if is_continuation:
                    item = dict(item)
                    item["data"] = item["data"] + nxt["data"]
                    merged.append(item)
                    merged.extend(items[i + 1:j])
                    i = j + 1
                    continue
        merged.append(item)
        i += 1
    return merged


def build_tree(items):
    root = SectionNode("root", "Document")
    stack, current_node = [root], root
    for item in items:
        if item["type"] == "text":
            text  = item["text"]
            match = SECTION_REGEX.match(text)
            if match:
                section_id = match.group(1)
                rest       = match.group(2).strip().lstrip(".").strip()
                if ":" in rest:
                    title, remaining = (p.strip() for p in rest.split(":", 1))
                else:
                    title, remaining = rest.strip(), ""
                title = title.strip(" .:-")
                if title and is_valid_section(title):
                    level    = get_level(section_id)
                    new_node = SectionNode(section_id, title)
                    if remaining:
                        new_node.add_content(remaining)
                    parent = find_parent(stack, section_id)
                    parent.children.append(new_node)
                    stack = [n for n in stack
                             if n.section_id == "root" or get_level(n.section_id) < level]
                    stack.append(new_node)
                    current_node = new_node
                    continue
            current_node.add_content(text)
        elif item["type"] == "table":
            current_node.add_table({"table_id": item["table_id"], "data": item["data"]})
    return root


def flatten_sections(node, parent=None, parent_map=None, depth=0):
    if parent_map is None:
        parent_map = {}
    chunks = []
    if node.section_id != "root":
        if parent:
            parent_map[node.section_id] = parent
        base_metadata = {
            "section_id":     node.section_id,
            "parent":         parent.section_id if parent else None,
            "hierarchy_path": get_hierarchy_path(node, parent_map),
            "depth":          depth,
            "title":          node.title,
            "has_tables":     len(node.tables) > 0,
            "num_tables":     len(node.tables),
        }
        content_text = " ".join(node.content).strip()
        section_text = f"{node.title}\n{content_text}" if content_text else node.title
        content_stats = {
            "content_length":  len(content_text),
            "word_count":      len(content_text.split()),
            "sentence_count":  len(re.split(r'[.!?]+', content_text)),
        }
        if section_text.strip():
            content_chunks = (split_long_content(content_text, 800, 150)
                              if len(content_text) > 800 else [content_text])
            for idx, cc in enumerate(content_chunks):
                cid = (f"{node.section_id}_text_{idx}"
                       if len(content_chunks) > 1 else f"{node.section_id}_text")
                chunks.append({
                    "id":      cid,
                    "section": node.section_id,
                    "title":   node.title,
                    "text":    cc,
                    "metadata": {
                        **base_metadata,
                        "chunk_type":            "section_content",
                        "is_continuation":       idx > 0,
                        "chunk_index":           idx,
                        "total_chunks":          len(content_chunks),
                        **content_stats,
                        "table_count_in_section": len(node.tables),
                    },
                })
        for t_idx, table in enumerate(node.tables):
            table_id   = table.get("table_id", f"table_{t_idx}")
            table_data = table.get("data", [])
            table_desc = generate_table_description(table_id, table_data)
            table_text = f"{node.title}\n{table_desc}\n"
            for row in table_data[:5]:
                row_str = " | ".join(str(cell).strip() for cell in row if cell)
                if row_str.strip():
                    table_text += f"{row_str}\n"
            if len(table_data) > 5:
                table_text += f"... ({len(table_data) - 5} more rows)"
            chunks.append({
                "id":         f"{node.section_id}_table_{t_idx}",
                "section":    node.section_id,
                "title":      node.title,
                "text":       table_text,
                "table_data": table_data,
                "metadata": {
                    **base_metadata,
                    "chunk_type":              "table",
                    "table_id":                table_id,
                    "table_index":             t_idx,
                    "total_tables_in_section": len(node.tables),
                    "table_rows":              len(table_data),
                    "table_cols":              len(table_data[0]) if table_data else 0,
                    "table_description":       table_desc,
                    **content_stats,
                },
            })
    for child in node.children:
        chunks.extend(flatten_sections(child, node, parent_map, depth + 1))
    return chunks


def parse_pdf(pdf_path: str) -> list:
    doc   = fitz.open(pdf_path)
    items = extract_page_items(doc)
    items = merge_split_tables(items)
    tree  = build_tree(items)
    return flatten_sections(tree)


print("✅ Parser core functions defined")

In [ ]:
# ── Run Parser ─────────────────────────────────────────────────
print(f"📖 Parsing: {PDF_PATH}")
chunks = parse_pdf(PDF_PATH)

# Save
with open(PARSED_JSON, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

# Statistics
section_chunks = [c for c in chunks if c['metadata'].get('chunk_type') == 'section_content']
table_chunks   = [c for c in chunks if c['metadata'].get('chunk_type') == 'table']
total_content  = sum(c['metadata'].get('content_length', 0) for c in section_chunks)
depth_dist     = {}
for c in chunks:
    d = c['metadata'].get('depth', 0)
    depth_dist[d] = depth_dist.get(d, 0) + 1

print(f"\n{'='*60}")
print("PARSER OUTPUT — v3 (Semantic Chunking & Rich Metadata)")
print(f"{'='*60}")
print(f"  Total Chunks       : {len(chunks)}")
print(f"  Section Chunks     : {len(section_chunks)}")
print(f"  Table Chunks       : {len(table_chunks)}")
print(f"  Total Content Size : {total_content:,} characters")
print(f"  Avg Chunk Size     : {total_content // max(len(section_chunks), 1):,} chars")
print(f"\nDepth Distribution:")
for depth in sorted(depth_dist):
    print(f"  Level {depth}: {depth_dist[depth]} chunks")
print(f"\n✅ Saved {len(chunks)} chunks → {PARSED_JSON}")

# Preview first 3 chunks
print("\n--- Preview (first 3 chunks) ---")
for idx, c in enumerate(chunks[:3]):
    print(f"\n[CHUNK {idx}]")
    print(f"  ID           : {c.get('id')}")
    print(f"  Section      : {c.get('section')}")
    print(f"  Chunk Type   : {c['metadata'].get('chunk_type')}")
    print(f"  Hierarchy    : {c['metadata'].get('hierarchy_path')}")
    print(f"  Text Preview : {c.get('text', '')[:120].replace(chr(10), ' ')}...")

---
## Cell 5 — Stage 2: Embed & Store (embed_and_store.py v4)

Loads parsed chunks, enriches metadata, generates sentence-transformer embeddings, and persists everything in a local ChromaDB vector store.

> **Note:** If you modify the parser or want a fresh store, delete `chroma_db/` before running this cell.

In [ ]:
# ══════════════════════════════════════════════════════════════
# EMBED & STORE — Helper functions
# ══════════════════════════════════════════════════════════════
SECTION_KEYWORDS = {
    "purpose":       ["purpose", "scope", "objective", "aim", "goal", "intent"],
    "procedure":     ["mix", "batch", "curing", "compaction", "placing", "pour", "casting",
                      "vibration", "mixing", "transport", "formwork", "reinforcement"],
    "equipment":     ["equipment", "mixer", "vibrator", "pump", "machine", "plant", "tool",
                      "formwork", "scaffold", "props", "concrete mixer", "needle"],
    "quality":       ["test", "strength", "quality", "cube", "sampling", "inspection",
                      "acceptance", "compressive", "crushing", "ndt", "slump", "workability"],
    "references":    ["is code", "bis", "standard", "specification", "reference", "doc",
                      "code", "is 456", "is 383", "is 9103"],
    "personnel":     ["personnel", "engineer", "supervisor", "inspector", "technician",
                      "responsible", "site engineer", "quality control", "foreman"],
    "health_safety": ["safety", "ppe", "hazard", "protection", "health", "environment",
                      "hse", "weather", "frost", "rain", "temperature"],
    "acronyms":      ["acronym", "abbreviation", "definition", "term", "terminology"],
    "scope":         ["scope", "coverage", "applies", "applicable", "applicability", "covers"],
}


def classify_section_enhanced(title: str, content: str, chunk_type: str,
                               hierarchy_path: str) -> str:
    full_context = f"{hierarchy_path} {title} {content}".lower()
    scores = {cat: sum(1 for kw in kws if kw in full_context)
              for cat, kws in SECTION_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    if scores[best] > 0:
        return best
    return "procedure" if chunk_type == "table" else "other"


def enrich_chunk_metadata(chunk: dict) -> dict:
    metadata       = chunk.get("metadata", {})
    content_length = metadata.get("content_length", 0)
    word_count     = metadata.get("word_count", 0)
    content_density = (word_count / max(content_length / 100, 1)) if content_length else 0
    if metadata.get("chunk_type") == "table":
        importance = 1.2
    else:
        depth      = metadata.get("depth", 0)
        importance = max(0.5, 1.0 / (1 + depth * 0.2))
    return {
        **metadata,
        "content_density":      round(content_density, 2),
        "importance_score":     round(importance, 2),
        "is_continuation_chunk": metadata.get("is_continuation", False),
        "chunk_position":       f"{metadata.get('chunk_index', 0)}/{metadata.get('total_chunks', 1)}",
    }


def build_embedding_text(chunk: dict) -> str:
    parts    = []
    metadata = chunk.get("metadata", {})
    hier     = metadata.get("hierarchy_path", "")
    if hier:
        parts.append(f"[Section: {hier}]")
    title = chunk.get("title", "")
    if title:
        parts.append(f"Title: {title}")
    text = chunk.get("text", "")
    if text:
        parts.append(text)
    if metadata.get("chunk_type") == "table":
        td = metadata.get("table_description", "")
        if td:
            parts.append(f"[Table Info: {td}]")
    return "\n".join(parts)


def is_v3_format(data: list) -> bool:
    if not data: return False
    first = data[0]
    return ("id" in first and "metadata" in first and
            "chunk_type" in first.get("metadata", {}))


def ensure_unique_ids(chunks: list) -> list:
    seen, processed = {}, []
    for chunk in chunks:
        oid = chunk.get("id", "chunk")
        if oid in seen:
            nid = f"{oid}_{seen[oid]}"
            seen[oid] += 1
        else:
            nid = oid
            seen[oid] = 1
        chunk["id"] = nid
        processed.append(chunk)
    return processed


def build_chunks(sections: list) -> list:
    print(f"📦 Detected {'v3' if is_v3_format(sections) else 'v2'} format")
    enriched = []
    for chunk in sections:
        metadata = chunk.get("metadata", {})
        label    = classify_section_enhanced(
            title=chunk.get("title", ""),
            content=chunk.get("text", ""),
            chunk_type=metadata.get("chunk_type", "section_content"),
            hierarchy_path=metadata.get("hierarchy_path", ""),
        )
        enriched_meta = enrich_chunk_metadata(chunk)
        enriched_meta["label"] = label
        enriched.append({
            "id":            chunk.get("id", f"chunk_{len(enriched)}"),
            "text":          build_embedding_text(chunk),
            "original_text": chunk.get("text", ""),
            "metadata":      enriched_meta,
        })
    print(f"\n🔍 Deduplicating chunk IDs...")
    return ensure_unique_ids(enriched)


print("✅ Embed & store helpers defined")

In [ ]:
# ── Run Embed & Store ──────────────────────────────────────────
print("\n" + "="*60)
print("EMBED & STORE PIPELINE (v4)")
print("="*60)

# Optional: clear existing DB for a fresh run
# import shutil; shutil.rmtree(DB_DIR, ignore_errors=True)

print(f"\n📖 Loading parsed sections from: {PARSED_JSON}")
with open(PARSED_JSON, "r", encoding="utf-8") as f:
    sections = json.load(f)
print(f"✅ Loaded {len(sections)} sections/chunks")

print(f"\n🔧 Building and enriching chunks...")
embed_chunks = build_chunks(sections)
print(f"✅ Built {len(embed_chunks)} enhanced chunks")

# Statistics
sc = [c for c in embed_chunks if c['metadata'].get('chunk_type') == 'section_content']
tc = [c for c in embed_chunks if c['metadata'].get('chunk_type') == 'table']
label_dist = {}
for c in embed_chunks:
    l = c['metadata'].get('label', 'unknown')
    label_dist[l] = label_dist.get(l, 0) + 1
print(f"\n📊 Chunk Statistics:")
print(f"   Total: {len(embed_chunks)}  |  Section: {len(sc)}  |  Table: {len(tc)}")
print(f"\n🏷️  Label Distribution:")
for label in sorted(label_dist):
    pct = label_dist[label] / len(embed_chunks) * 100
    print(f"   {label:15s}: {label_dist[label]:4d} ({pct:5.1f}%)")

# Embedding
print(f"\n🧠 Loading embedding model: {EMBEDDING_MODEL_NAME}")
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"✅ Model loaded (dim={embed_model.get_sentence_embedding_dimension()})")

print(f"\n⚙️  Generating embeddings for {len(embed_chunks)} chunks...")
texts = [c["text"] for c in embed_chunks]
embeddings = embed_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
embeddings_list = embeddings.tolist()
print(f"✅ Generated {len(embeddings_list)} embeddings")

# ChromaDB storage
print(f"\n💾 Connecting to ChromaDB at: {DB_DIR}")
chroma_client = chromadb.PersistentClient(path=DB_DIR)
collection    = chroma_client.get_or_create_collection("sections")
print(f"✅ Connected to collection: sections")

# Validate no duplicate IDs
chunk_ids = [c["id"] for c in embed_chunks]
assert len(chunk_ids) == len(set(chunk_ids)), "Duplicate IDs found — check parsed_sections.json"

print(f"\n📝 Upserting {len(embed_chunks)} chunks into ChromaDB...")
collection.upsert(
    ids=[c["id"] for c in embed_chunks],
    embeddings=embeddings_list,
    documents=[c["original_text"] for c in embed_chunks],
    metadatas=[c["metadata"] for c in embed_chunks],
)
print(f"✅ Upserted {len(embed_chunks)} chunks")
print(f"\n{'='*60}")
print("✅ EMBEDDING & STORAGE COMPLETE")
print(f"{'='*60}")

---
## Cell 6 — Stage 3: Retrieve & Generate (retrieve_and_generate.py v5)

For each of the 10 Method Statement sections:
- Multi-query ChromaDB retrieval (prose + table passes)
- Cross-encoder reranking with importance-score boosting
- Groq LLM (`llama-3.3-70b-versatile`) generation
- Jaccard + BERT Score evaluation metrics

In [ ]:
# ══════════════════════════════════════════════════════════════
# SECTION DEFINITIONS (all 10 MS sections)
# ══════════════════════════════════════════════════════════════
MS_SECTIONS = [
    {
        "key": "purpose",
        "heading": "1. Purpose of the Method Statement",
        "queries": [
            "purpose scope objective of reinforced cement concrete RCC work",
            "general requirements for RCC construction specification",
            "intent application method statement RCC structural concrete works",
            "description overview reinforced concrete construction project requirement",
        ],
        "word_limit": 120,
        "instruction": (
            "Write a concise purpose statement (2–3 sentences) explaining WHY "
            "this method statement exists, referencing the RCC works covered by "
            "the specification."
        ),
    },
    {
        "key": "scope",
        "heading": "2. Scope of the Method Statement",
        "queries": [
            "scope of work reinforced cement concrete structural elements",
            "applicability RCC specification foundations columns beams slabs",
            "coverage extent types structural members concrete grade specification",
            "inclusions exclusions locations applicable sections concrete works",
        ],
        "word_limit": 150,
        "instruction": (
            "Define WHAT work is covered — structural elements, locations, "
            "grade of concrete — citing the specification sections."
        ),
    },
    {
        "key": "acronyms",
        "heading": "3. Acronyms and Definitions",
        "queries": [
            "abbreviations acronyms definitions terminology concrete specification",
            "IS code BIS OPC PPC w/c ratio slump workability definitions",
        ],
        "word_limit": 200,
        "instruction": (
            "List all acronyms and technical terms found in the retrieved text "
            "as a concise definition list (ACRONYM – full form: succinct meaning).  "
            "CRITICAL formatting rules:\n"
            "  - Each entry must be fully self-contained: state what the acronym "
            "    stands for AND what it means in plain technical language.\n"
            "  - For SCC consistency/flow classes (VS1, VS2, VF1, VF2, SF1–SF3, "
            "    PA0–PA2, PL1–PL2, etc.) write the full class name, then describe "
            "    the property range or test value it represents.\n"
            "  - Do NOT use dangling clauses without first stating the full form.\n"
            "  - Do NOT invent definitions not supported by the retrieved text.\n"
            "  - Each definition should be 1–2 sentences max."
        ),
    },
    {
        "key": "references",
        "heading": "4. Reference Documents",
        "queries": [
            "IS code standards references BIS bureau Indian standard concrete",
            "relevant codes specifications standards cited in document",
            "IS 456 IS 383 IS 516 IS 1199 code number referenced specification",
            "standard specification number clause reference concrete reinforcement",
        ],
        "word_limit": 200,
        "instruction": (
            "List every IS code, BIS standard, or other document explicitly "
            "cited in the retrieved chunks.  Format as a numbered list."
        ),
    },
    {
        "key": "procedure",
        "heading": "5. Procedure for Concreting",
        "queries": [
            "procedure batching mixing transporting placing compacting curing concrete",
            "concrete mix design proportioning water cement ratio aggregate",
            "formwork shuttering reinforcement bar bending placing cover",
            "compaction vibration consolidation concrete placing procedure",
            "curing methods period water curing membrane curing concrete",
            "concrete pour sequence joints construction cold joints",
            "admixtures plasticizer retarder accelerator concrete mixing",
        ],
        "word_limit": 600,
        "instruction": (
            "Write a step-by-step concreting procedure with numbered sub-steps: "
            "(a) Formwork, (b) Reinforcement, (c) Mix Design / Batching, "
            "(d) Mixing, (e) Transportation & Placing, (f) Compaction, "
            "(g) Finishing, (h) Curing.  "
            "Include specific values (grades, w/c ratios, slump, curing period) "
            "from the specification where available."
        ),
    },
    {
        "key": "equipment",
        "heading": "6. Equipment Used",
        "queries": [
            "equipment machinery plant concrete mixer batching plant transit mixer pump",
            "vibrator needle poker vibration compaction equipment tools",
            "formwork shuttering props scaffolding equipment",
        ],
        "word_limit": 200,
        "instruction": (
            "List all plant, equipment, and tools mentioned in the specification "
            "for RCC work as a bullet list with a brief note on its use."
        ),
    },
    {
        "key": "personnel",
        "heading": "7. Key People Involved",
        "queries": [
            "personnel responsible engineer supervisor quality control inspection",
            "testing laboratory inspector site engineer foreman concrete",
        ],
        "word_limit": 150,
        "instruction": (
            "List roles/personnel mentioned or implied in the specification "
            "(e.g., site engineer, quality inspector, lab technician) and their "
            "responsibilities."
        ),
    },
    {
        "key": "quality",
        "heading": "8. Quality Control and Testing",
        "queries": [
            "quality control testing cube strength acceptance criteria concrete",
            "sampling frequency cube mould testing compressive strength",
            "inspection checks hold points concrete placement",
        ],
        "word_limit": 250,
        "instruction": (
            "Summarise quality-control requirements: sampling frequency, test "
            "types, acceptance criteria, and any hold/witness points from the "
            "specification."
        ),
    },
    {
        "key": "health_safety",
        "heading": "9. Health, Safety & Environment (HSE) Considerations",
        "queries": [
            "safety health environment precautions hazards concrete construction",
            "PPE protective equipment safety measures workers",
        ],
        "word_limit": 150,
        "instruction": (
            "Summarise HSE measures stated in the specification.  If none are "
            "explicitly stated, note that only and do NOT invent content."
        ),
    },
    {
        "key": "other",
        "heading": "10. Other Relevant Information",
        "queries": [
            "special requirements restrictions notes concrete specification",
            "rejection defective concrete remedial action non-conformance",
            "hot weather cold weather concreting special conditions",
        ],
        "word_limit": 150,
        "instruction": (
            "Include any other specification requirements not covered in the "
            "sections above (e.g. weather conditions, hot/cold weather "
            "concreting, repair of defective work)."
        ),
    },
]

print(f"✅ {len(MS_SECTIONS)} MS section definitions loaded")

In [ ]:
# ══════════════════════════════════════════════════════════════
# RETRIEVAL HELPERS
# ══════════════════════════════════════════════════════════════
def _build_query_text(query: str) -> str:
    """Mirror embed_and_store v4 build_embedding_text() prefix format."""
    return f"Title: {query}\n{query}"


def _query_collection(coll, embedding, n_results: int,
                      chunk_type_filter=None) -> list:
    kwargs = dict(
        query_embeddings=embedding,
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )
    if chunk_type_filter:
        kwargs["where"] = {"chunk_type": chunk_type_filter}
    try:
        res = coll.query(**kwargs)
    except Exception:
        kwargs["n_results"] = max(1, n_results // 2)
        kwargs.pop("where", None)
        try:
            res = coll.query(**kwargs)
        except Exception:
            return []
    rows = []
    for doc, meta, cid, dist in zip(
        res["documents"][0], res["metadatas"][0],
        res["ids"][0],       res["distances"][0],
    ):
        if dist <= DISTANCE_THRESHOLD:
            rows.append((cid, doc, meta, dist))
    return rows


def _importance_boost(meta: dict) -> float:
    score = meta.get("importance_score", 1.0)
    depth = meta.get("depth", 2)
    if depth == 0:   score += 0.15
    elif depth == 1: score += 0.07
    if meta.get("has_tables", False):
        score += 0.05
    if meta.get("is_continuation_chunk", False):
        score -= 0.05
    return max(score, 0.1)


def retrieve_chunks(coll, model, queries: list, n_per_query: int = 8) -> list:
    seen_ids, raw = set(), []
    for query in queries:
        q_text    = _build_query_text(query)
        embedding = model.encode([q_text]).tolist()
        for row in _query_collection(coll, embedding, n_per_query):
            if row[0] not in seen_ids:
                seen_ids.add(row[0])
                raw.append(row)
        for row in _query_collection(coll, embedding,
                                     max(2, n_per_query // 2), "table"):
            if row[0] not in seen_ids:
                seen_ids.add(row[0])
                raw.append(row)
    if not raw:
        return []
    reranker     = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    combined_q   = " ".join(queries)
    pairs        = [(combined_q, row[1]) for row in raw]
    ce_scores    = reranker.predict(pairs)
    boosted = []
    for i, row in enumerate(raw):
        cid, doc, meta, dist = row
        final_score = float(ce_scores[i]) * _importance_boost(meta)
        boosted.append((cid, doc, meta, dist, final_score))
    boosted.sort(key=lambda r: r[4], reverse=True)
    boosted = boosted[:RETRIEVAL_POOL_CAP]
    return [
        {
            "id":             cid,
            "section":        meta.get("section", ""),
            "title":          meta.get("title", ""),
            "chunk_type":     meta.get("chunk_type", "section_content"),
            "table_id":       meta.get("table_id", ""),
            "hierarchy_path": meta.get("hierarchy_path", ""),
            "depth":          meta.get("depth", 0),
            "label":          meta.get("label", ""),
            "importance":     meta.get("importance_score", 1.0),
            "has_tables":     meta.get("has_tables", False),
            "is_continuation":meta.get("is_continuation_chunk", False),
            "text":           doc,
            "distance":       round(dist, 4),
            "final_score":    round(boosted[i][4], 4),
        }
        for i, (cid, doc, meta, dist, _) in enumerate(boosted)
    ]


print("✅ Retrieval helpers defined")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONTEXT BUILDER + LLM
# ══════════════════════════════════════════════════════════════
def pipe_to_markdown_table(text: str) -> str:
    lines = [l for l in text.strip().splitlines() if l.strip()]
    if not lines: return text
    out, data_lines, caption = [], [], ""
    if "|" not in lines[0]:
        caption    = lines[0].strip()
        data_lines = lines[1:]
    else:
        data_lines = lines
    if caption:
        out.append(f"**{caption}**")
    if not data_lines:
        return "\n".join(out)
    def _row(cells): return "| " + " | ".join(c.strip() for c in cells) + " |"
    header_cells = data_lines[0].split("|")
    out.append(_row(header_cells))
    out.append("| " + " | ".join("---" for _ in header_cells) + " |")
    for row in data_lines[1:]:
        out.append(_row(row.split("|")))
    return "\n".join(out)


def chunks_to_context(chunks: list, max_chars: int = MAX_CONTEXT_CHARS) -> str:
    prose_blocks, table_blocks, total_chars = [], [], 0
    for i, c in enumerate(chunks, 1):
        chunk_type = c.get("chunk_type", "section_content")
        hier       = c.get("hierarchy_path", "")
        cont_note  = " [continued]" if c.get("is_continuation") else ""
        header     = (
            f"[{i}] §{c['section']} — {c['title']}{cont_note}"
            + (f"  (path: {hier})" if hier else "")
        )
        if chunk_type == "table":
            body  = pipe_to_markdown_table(c["text"])
            block = f"{header}\n{body}\n"
        else:
            block = f"{header}\n{c['text'].strip()}\n"
        if total_chars + len(block) > max_chars:
            break
        (table_blocks if chunk_type == "table" else prose_blocks).append(block)
        total_chars += len(block)
    parts = []
    if prose_blocks:
        parts.append("=== SPECIFICATION TEXT ===\n" + "\n".join(prose_blocks))
    if table_blocks:
        parts.append("=== SPECIFICATION TABLES ===\n" + "\n".join(table_blocks))
    return "\n\n".join(parts)


SYSTEM_PROMPT = """\
You are a technical writer creating a Method Statement for Reinforced Cement \
Concrete (RCC) works on a construction project.

RULES:
1. Use ONLY information from the specification excerpts provided.
2. If the excerpts do not contain enough information for a section, write:
   "Not explicitly stated in the specification."
3. Do NOT hallucinate values, codes, or requirements.
4. Be concise and professional.
5. Cite the specification section number (e.g. "per §4.1.2") wherever possible.
6. The context contains two blocks:
   - SPECIFICATION TEXT  : prose paragraphs from the spec (hierarchy path shown)
   - SPECIFICATION TABLES: data tables in markdown format
   When referencing table values (sieve sizes, mix ratios, % passing etc.)
   read the column header and row label carefully before quoting a number.
7. Chunk headers show the hierarchy path (e.g. "5 > 5.1 > 5.1.2") — use this
   to understand the structural context of each excerpt.
8. Quote key technical terms, values, and phrases DIRECTLY from the specification
   excerpts wherever possible (e.g. grades, ratios, period values, code numbers).
   Preserve the exact wording of requirements rather than paraphrasing them.
"""


def call_groq(api_key: str, context: str, section_instruction: str,
              word_limit: int, model_name: str = "llama-3.3-70b-versatile") -> str:
    client   = Groq(api_key=api_key)
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"--- SPECIFICATION EXCERPTS ---\n{context}\n\n"
                f"--- TASK ---\n{section_instruction}\n"
                f"Word limit: approximately {word_limit} words.\n\n"
                f"STEP 1 — Before writing, silently identify the 2-3 most relevant "
                f"sentences or phrases from the excerpts above that directly answer "
                f"this section.  Keep them in mind as verbatim anchors.\n"
                f"STEP 2 — Write the section content now (no heading, just the body "
                f"text).  Use at least 40% direct phrases and key terms taken "
                f"verbatim from the retrieved specification text.  "
                f"Do NOT add a 'STEP 1' header in your output — output only the "
                f"final section body:"
            )},
        ],
        max_tokens=1024,
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()


print("✅ Context builder and LLM helpers defined")

In [ ]:
# ══════════════════════════════════════════════════════════════
# ACCURACY METRICS
# ══════════════════════════════════════════════════════════════
def compute_retrieval_coverage(sections_content: dict, retrieved_chunks: dict) -> dict:
    def tokenise(text):
        STOPWORDS = {"the","a","an","of","in","to","and","or","is","are",
                     "be","as","for","with","by","on","at","from","it","its",
                     "not","per","shall"}
        tokens = set(re.findall(r"[a-z0-9]+", text.lower()))
        return tokens - STOPWORDS - {t for t in tokens if len(t) <= 1}

    def jaccard(a, b):
        if not a or not b: return 0.0
        inter = len(a & b)
        union = len(a | b)
        return round(inter / union, 4) if union else 0.0

    scores = {}
    for key, chunks in retrieved_chunks.items():
        gen_tokens = tokenise(sections_content.get(key, ""))
        if not chunks:
            scores[key] = {"jaccard": None, "n_chunks": 0,
                           "n_table_chunks": 0, "best_chunk": None}
            continue
        jaccard_scores, best_score, best_chunk = [], -1, None
        for c in chunks:
            j = jaccard(gen_tokens, tokenise(c["text"]))
            jaccard_scores.append(j)
            if j > best_score:
                best_score, best_chunk = j, c["section"]
        mean_j  = round(sum(jaccard_scores) / len(jaccard_scores), 4)
        n_table = sum(1 for c in chunks if c.get("chunk_type") == "table")
        scores[key] = {"jaccard": mean_j, "n_chunks": len(chunks),
                       "n_table_chunks": n_table, "best_chunk": best_chunk}
    return scores


def compute_bert_scores(sections_content: dict, retrieved_chunks: dict) -> dict:
    if not BERT_SCORE_AVAILABLE:
        return {k: {"bert_f1": None, "bert_p": None, "bert_r": None}
                for k in sections_content}
    results = {}
    for key, gen_text in sections_content.items():
        chunks = retrieved_chunks.get(key, [])
        if not chunks or not gen_text.strip():
            results[key] = {"bert_f1": None, "bert_p": None, "bert_r": None}
            continue
        reference = " ".join(c["text"] for c in chunks if c.get("text"))
        if not reference.strip():
            results[key] = {"bert_f1": None, "bert_p": None, "bert_r": None}
            continue
        try:
            P, R, F1 = bert_score_fn([gen_text], [reference], lang="en", verbose=False)
            results[key] = {"bert_f1": round(float(F1[0]), 4),
                            "bert_p":  round(float(P[0]),  4),
                            "bert_r":  round(float(R[0]),  4)}
        except Exception as e:
            results[key] = {"bert_f1": None, "bert_p": None, "bert_r": None,
                            "error": str(e)}
    return results


print("✅ Metrics helpers defined")

In [ ]:
# ── Run Retrieve & Generate Pipeline ──────────────────────────
assert GROQ_API_KEY and GROQ_API_KEY != "YOUR_GROQ_API_KEY_HERE", \
    "Set GROQ_API_KEY in the Configuration cell above. Get a free key at https://console.groq.com"

# Reload collection (in case kernel was restarted after embedding)
chroma_client2 = chromadb.PersistentClient(path=DB_DIR)
collection2    = chroma_client2.get_or_create_collection("sections")
total_docs     = collection2.count()
print(f"⏳ Loading embedding model ({EMBEDDING_MODEL_NAME}) …")
embed_model2   = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"✅ ChromaDB: {total_docs} chunks | Embedding model loaded")

sections_content, all_retrieved = {}, {}

for sec in MS_SECTIONS:
    key     = sec["key"]
    heading = sec["heading"]
    print(f"\n📄 Processing: {heading}")

    # Stage 4 — retrieve
    chunks = retrieve_chunks(collection2, embed_model2, sec["queries"], N_PER_QUERY)
    all_retrieved[key] = chunks

    n_tables = sum(1 for c in chunks if c.get("chunk_type") == "table")
    n_cont   = sum(1 for c in chunks if c.get("is_continuation"))
    print(f"   → {len(chunks)} chunks  (tables={n_tables}, continuations={n_cont})")
    for c in chunks[:3]:
        print(f"      ↳ §{c['section']} [{c.get('chunk_type','?')}] "
              f"path={c.get('hierarchy_path','?')}  score={c.get('final_score',0):.3f}")

    context = chunks_to_context(chunks)

    # Stage 5 — generate
    print(f"   → Calling Groq (llama-3.3-70b-versatile) …")
    try:
        text = call_groq(
            api_key             = GROQ_API_KEY,
            context             = context,
            section_instruction = sec["instruction"],
            word_limit          = sec["word_limit"],
        )
    except Exception as e:
        print(f"   ⚠  Groq error: {e}")
        text = f"Generation failed: {e}"

    sections_content[key] = text
    print(f"   → {len(text.split())} words generated")

# Save debug JSON
debug_path = os.path.join(OUT_DIR, "generated_sections_debug.json")
with open(debug_path, "w", encoding="utf-8") as f:
    json.dump(sections_content, f, ensure_ascii=False, indent=2)

chunks_debug_path = os.path.join(OUT_DIR, "retrieved_chunks_debug.json")
with open(chunks_debug_path, "w", encoding="utf-8") as f:
    json.dump(all_retrieved, f, ensure_ascii=False, indent=2)

print(f"\n✅ Debug JSON → {debug_path}")
print(f"✅ Retrieved chunks → {chunks_debug_path}")

In [ ]:
# ── Accuracy Metrics ───────────────────────────────────────────
jaccard_scores = compute_retrieval_coverage(sections_content, all_retrieved)
print("\n📊 Jaccard token-overlap scores (0→1):")
for k, v in jaccard_scores.items():
    if v["jaccard"] is None:
        print(f"   {k:<15}: no chunks retrieved")
    else:
        print(f"   {k:<15}: {v['jaccard']:.4f}  "
              f"(chunks={v['n_chunks']}, tables={v['n_table_chunks']}, "
              f"best=§{v['best_chunk']})")

print("\n📊 BERTScore (F1 / P / R vs retrieved context):")
bert_scores = compute_bert_scores(sections_content, all_retrieved)
for k, v in bert_scores.items():
    if v.get("bert_f1") is None:
        print(f"   {k:<15}: N/A  ({v.get('error', 'not available')})")
    else:
        print(f"   {k:<15}: F1={v['bert_f1']:.4f}  P={v['bert_p']:.4f}  R={v['bert_r']:.4f}")

combined_metrics = {k: {**jaccard_scores[k], **bert_scores.get(k, {})} for k in jaccard_scores}
valid_j   = [v["jaccard"]  for v in jaccard_scores.values() if v["jaccard"]       is not None]
valid_b   = [v["bert_f1"]  for v in bert_scores.values()    if v.get("bert_f1")   is not None]
avg_j     = round(sum(valid_j) / len(valid_j), 4) if valid_j else None
avg_b     = round(sum(valid_b) / len(valid_b), 4) if valid_b else None

combined_metrics["_summary"] = {
    "avg_jaccard":      avg_j,
    "avg_bert_f1":      avg_b,
    "embedding_model":  EMBEDDING_MODEL_NAME,
    "llm_model":        "llama-3.3-70b-versatile (Groq)",
    "n_sections":       len(MS_SECTIONS),
}

print(f"\n📈 Summary — avg Jaccard: {avg_j}  |  avg BERT F1: {avg_b}")

metric_path = os.path.join(OUT_DIR, "accuracy_metrics.json")
with open(metric_path, "w") as f:
    json.dump(combined_metrics, f, indent=2)
print(f"✅ Metrics saved → {metric_path}")

---
## Cell 7 — Stage 4: Generate DOCX

Python equivalent of `generate_docx.js` using `python-docx`.
Produces a styled Word document with:
- Cover page (team name, members, date, document ref)
- 10 Method Statement sections with branded headings
- Header/footer with page numbers
- Notes page (pipeline parameters, document conventions)

In [ ]:
# ══════════════════════════════════════════════════════════════
# DOCX BUILDER — Design tokens & helpers
# ══════════════════════════════════════════════════════════════
from docx import Document as DocxDoc
from docx.shared import Pt, RGBColor, Inches, Cm, Twips
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from copy import deepcopy
import lxml.etree as etree

# Brand colours (R, G, B)
C_BRAND        = RGBColor(0x1F, 0x4E, 0x79)   # deep navy
C_ACCENT       = RGBColor(0x2E, 0x75, 0xB6)   # mid-blue
C_ACCENT_LIGHT = RGBColor(0xD6, 0xE4, 0xF0)   # pale blue
C_BODY         = RGBColor(0x1A, 0x1A, 0x1A)   # near-black
C_MUTED        = RGBColor(0x59, 0x59, 0x59)   # grey
C_WHITE        = RGBColor(0xFF, 0xFF, 0xFF)

FONT_BODY    = "Calibri"
FONT_HEADING = "Calibri"


def _set_para_spacing(para, before=0, after=0, line=None):
    pf = para.paragraph_format
    pf.space_before = Pt(before)
    pf.space_after  = Pt(after)
    if line:
        from docx.shared import Pt as _Pt
        pf.line_spacing = _Pt(line)


def _add_run(para, text, bold=False, italic=False, size=11,
             color=None, font=None, all_caps=False):
    run        = para.add_run(text)
    run.bold   = bold
    run.italic = italic
    run.font.size = Pt(size)
    if color:  run.font.color.rgb = color
    if font:   run.font.name = font
    if all_caps:
        run._element.get_or_add_rPr().append(
            OxmlElement('w:caps'))
    return run


def _add_horizontal_rule(doc, color_hex="BDD7EE", thickness=6):
    """Add a paragraph with a bottom border acting as a horizontal rule."""
    para = doc.add_paragraph()
    _set_para_spacing(para, 0, 0)
    pPr  = para._p.get_or_add_pPr()
    pBdr = OxmlElement('w:pBdr')
    bottom = OxmlElement('w:bottom')
    bottom.set(qn('w:val'), 'single')
    bottom.set(qn('w:sz'),  str(thickness))
    bottom.set(qn('w:color'), color_hex)
    bottom.set(qn('w:space'), '1')
    pBdr.append(bottom)
    pPr.append(pBdr)
    return para


def _add_section_heading(doc, text):
    """Heading 1 style: bold navy + underline rule below."""
    doc.add_paragraph()  # small spacer
    para = doc.add_heading(text, level=1)
    para.clear()  # clear default run
    _set_para_spacing(para, 4, 2)
    run = para.add_run(text)
    run.bold          = True
    run.font.name     = FONT_HEADING
    run.font.size     = Pt(13)
    run.font.color.rgb = C_BRAND
    _add_horizontal_rule(doc, "2E75B6", 6)


def _add_sub_heading(doc, text):
    para = doc.add_heading(text, level=2)
    para.clear()
    _set_para_spacing(para, 6, 2)
    run = para.add_run(text)
    run.bold           = True
    run.font.name      = FONT_HEADING
    run.font.size      = Pt(12)
    run.font.color.rgb = C_ACCENT


def _add_body_para(doc, text, bold=False, align=WD_ALIGN_PARAGRAPH.JUSTIFY):
    para = doc.add_paragraph()
    _set_para_spacing(para, 1, 2, line=13)
    para.alignment = align
    _add_run(para, text, bold=bold, size=11, color=C_BODY, font=FONT_BODY)
    return para


def _add_bullet(doc, text):
    para = doc.add_paragraph(style='List Bullet')
    _set_para_spacing(para, 1, 2)
    _add_run(para, text, size=11, color=C_BODY, font=FONT_BODY)


def _add_numbered(doc, text):
    para = doc.add_paragraph(style='List Number')
    _set_para_spacing(para, 1, 2)
    _add_run(para, text, size=11, color=C_BODY, font=FONT_BODY)


print("✅ DOCX helpers defined")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONTENT PARSER — converts LLM output to styled DOCX paragraphs
# ══════════════════════════════════════════════════════════════
import re as _re

BULLET_RE   = _re.compile(r'^[-*•]\s+')
NUMBERED_RE = _re.compile(r'^\d+[.)]\s+')
BOLD_LINE_RE = _re.compile(r'^\*\*(.*?)\*\*\s*$')
INLINE_BOLD  = _re.compile(r'\*\*(.*?)\*\*')


def _add_rich_para(doc, text, style=None, align=WD_ALIGN_PARAGRAPH.JUSTIFY,
                   spacing_before=1, spacing_after=2):
    """Add a paragraph with inline **bold** support."""
    if style:
        para = doc.add_paragraph(style=style)
    else:
        para = doc.add_paragraph()
    _set_para_spacing(para, spacing_before, spacing_after, line=13)
    para.alignment = align

    last = 0
    for m in INLINE_BOLD.finditer(text):
        if m.start() > last:
            _add_run(para, text[last:m.start()], size=11, color=C_BODY, font=FONT_BODY)
        _add_run(para, m.group(1), bold=True, size=11, color=C_BODY, font=FONT_BODY)
        last = m.end()
    if last < len(text):
        _add_run(para, text[last:], size=11, color=C_BODY, font=FONT_BODY)
    return para


def parse_and_add_content(doc, raw: str):
    """Parse LLM output text and add styled paragraphs to the document."""
    for line in raw.split("\n"):
        line = line.rstrip()
        if not line.strip():
            doc.add_paragraph()
            continue
        if BOLD_LINE_RE.match(line):
            head_text = line.strip().strip("**").strip()
            _add_sub_heading(doc, head_text)
            continue
        stripped = line.strip()
        if BULLET_RE.match(stripped):
            text = BULLET_RE.sub("", stripped)
            _add_rich_para(doc, text, style='List Bullet')
            continue
        if NUMBERED_RE.match(stripped):
            text = NUMBERED_RE.sub("", stripped)
            _add_rich_para(doc, text, style='List Number')
            continue
        _add_rich_para(doc, stripped)


print("✅ Content parser defined")

In [ ]:
# ══════════════════════════════════════════════════════════════
# COVER PAGE + NOTES PAGE
# ══════════════════════════════════════════════════════════════
def build_cover_page(doc, team_name, team_members):
    today = datetime.now().strftime("%d %B %Y")

    # Title block
    title_para = doc.add_paragraph()
    title_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    _set_para_spacing(title_para, 10, 4)
    run = title_para.add_run("METHOD STATEMENT")
    run.bold = True
    run.font.name = FONT_HEADING
    run.font.size = Pt(28)
    run.font.color.rgb = C_BRAND

    sub_para = doc.add_paragraph()
    sub_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    _set_para_spacing(sub_para, 0, 6)
    run2 = sub_para.add_run("Reinforced Cement Concrete (RCC) Works")
    run2.font.name = FONT_HEADING
    run2.font.size = Pt(16)
    run2.font.color.rgb = C_ACCENT

    _add_horizontal_rule(doc, "2E75B6", 10)
    doc.add_paragraph()

    # Meta table
    tbl = doc.add_table(rows=0, cols=2)
    tbl.style = 'Table Grid'
    # Remove all borders
    for row in tbl.rows:
        for cell in row.cells:
            for border in ['top','bottom','left','right']:
                tc = cell._tc
                tcPr = tc.get_or_add_tcPr()
                tcBdr = OxmlElement('w:tcBdr')
                b = OxmlElement(f'w:{border}')
                b.set(qn('w:val'), 'none')
                tcBdr.append(b)
                tcPr.append(tcBdr)

    def add_meta_row(label, value_lines):
        row   = tbl.add_row()
        cell0 = row.cells[0]
        cell1 = row.cells[1]
        p0 = cell0.paragraphs[0]
        _add_run(p0, label, bold=True, size=11, color=C_BRAND, font=FONT_BODY)
        for i, val in enumerate(value_lines):
            p1 = cell1.paragraphs[0] if i == 0 else cell1.add_paragraph()
            _add_run(p1, val, size=11, color=C_BODY, font=FONT_BODY)

    add_meta_row("Prepared by:",   [team_name])
    add_meta_row("Team Members:",  team_members)
    add_meta_row("Date:",          [today])
    add_meta_row("Document Ref.:", ["Based on CPWD Prescriptive Specifications"])

    doc.add_paragraph()
    _add_horizontal_rule(doc, "BDD7EE", 4)
    conf_para = doc.add_paragraph()
    conf_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    _add_run(conf_para, "CONFIDENTIAL — FOR PROJECT USE ONLY",
             italic=True, size=9, color=C_MUTED, font=FONT_BODY)

    # Page break
    doc.add_page_break()


def build_notes_page(doc):
    _add_section_heading(doc, "Document Conventions")
    _add_body_para(doc,
        "Throughout this document, the symbol '§' (section sign) refers to specific "
        "clauses or subsections within the CPWD Prescriptive Specifications. "
        "For example, '§5.1.3' indicates Clause 5.1.3 of the specification document."
    )

    _add_section_heading(doc, "Note on Sources")
    _add_body_para(doc,
        "All information in this Method Statement has been extracted from the "
        "CPWD Prescriptive Specifications document using an NLP-based retrieval "
        "pipeline (parser v3 + embed_and_store v4 + retrieve_and_generate v5).  "
        "Where specification text was insufficient, this has been explicitly noted "
        "within the relevant section."
    )

    _add_horizontal_rule(doc, "BDD7EE", 4)

    label_para = doc.add_paragraph()
    _add_run(label_para, "Pipeline parameters:", bold=True, size=10,
             color=C_MUTED, font=FONT_BODY)

    params = [
        ("Embedding model",  EMBEDDING_MODEL_NAME),
        ("LLM",              "llama-3.3-70b-versatile (Groq)"),
        ("BERT eval model",  "roberta-large (rescaled baseline)"),
        ("Retrieval depth",  f"{RETRIEVAL_POOL_CAP} chunks / section (cross-encoder reranked)"),
        ("Context window",   f"{MAX_CONTEXT_CHARS:,} characters"),
        ("Temperature",      "0.2"),
    ]
    for k, v in params:
        p = doc.add_paragraph()
        _set_para_spacing(p, 1, 1)
        _add_run(p, f"{k}", bold=True, size=11, color=C_BRAND, font=FONT_BODY)
        _add_run(p, f"  —  {v}", size=11, color=C_BODY, font=FONT_BODY)


print("✅ Cover page and notes page functions defined")

In [ ]:
# ── Build & Save DOCX ─────────────────────────────────────────
print(f"🎨 Building DOCX document …")

doc = DocxDoc()

# Page size: A4
from docx.oxml.ns import qn
section_props = doc.sections[0]
section_props.page_width  = Cm(21)
section_props.page_height = Cm(29.7)
section_props.top_margin    = Inches(0.75)
section_props.bottom_margin = Inches(0.75)
section_props.left_margin   = Inches(1.0)
section_props.right_margin  = Inches(1.0)

# Cover page
build_cover_page(doc, TEAM_NAME, TEAM_MEMBERS)

# Body: 10 sections
for sec in MS_SECTIONS:
    _add_section_heading(doc, sec["heading"])
    content = sections_content.get(sec["key"], "Content not generated.")
    parse_and_add_content(doc, content)

# Notes page
doc.add_page_break()
build_notes_page(doc)

# Save
doc.save(DOCX_OUT)
print(f"✅ Word document saved → {DOCX_OUT}")

---
## Cell 8 — Download Outputs (Colab)

Downloads the generated `.docx`, debug JSON, and metrics JSON to your local machine.

In [ ]:
try:
    from google.colab import files
    print("📥 Downloading output files …")
    files.download(DOCX_OUT)
    files.download(debug_path)
    files.download(metric_path)
    print("✅ Downloads triggered.")
except ImportError:
    print(f"Running locally — output files are at:")
    print(f"  DOCX    : {DOCX_OUT}")
    print(f"  Sections: {debug_path}")
    print(f"  Metrics : {metric_path}")